# Reversible Language-Model Training

**ERA V5 Session 13 — Distributed Training II, model parallelism, and reversible networks**

This notebook is the main Colab entry point. Helper modules live in `helpers/` and are the same code the pytest suite validates. Full 50M-token runs need a CUDA GPU. CPU runtimes can execute smoke tests only.

Primary research questions:

1. Can a ~20M-parameter decoder-only Transformer be trained on 50M target tokens with a standard residual stack and with a **midpoint / leapfrog** reversible stack that stores only boundary states?
2. At a **fixed micro-batch**, what is the memory-versus-compute trade-off of algebraic reconstruction versus storing activations?
3. Does reversibility unlock a **larger reliable micro-batch**, and does that change loss, throughput, or peak memory in a way that is attributable to reversibility rather than to batch size?

This notebook does not claim that reversible training must win. It reports whatever the selected hardware actually produces.


## 1. Assignment objective and research questions

The proven S13 protocol is **four runs** on a 50M-token budget:

| Run | Model | Micro-batch |
| --- | --- | --- |
| 1 / **A** | Ordinary residual Transformer | **32** |
| 2 / **B euler** | Euler position+velocity, `h = 1/sqrt(L)` | **32** |
| 3 / **B midpoint** | Midpoint / leapfrog, `h = 0.5` | **32** |
| 4 / **C** | Winner of Euler vs midpoint | **128**, then 64, then 32 on OOM |

Token convention: a packed (or random-window) example of length `T` contributes `T` non-padding target-token positions. Validation tokens are counted separately.

`memcnn` / `revtorch` are not required. Both lecture integrators are implemented in PyTorch.


## 2. Ordinary versus reversible training

A standard Pre-LN block is the first-order residual

$$x_{l+1} = x_l + F_l(x_l)$$

Backprop stores every $F_l$ activation (or checkpoints them and reruns $F_l$). Checkpointing is **not** reversibility.

Both reversible fixes carry **two grids**. Hidden states stay **fp32** because reversibility subtracts; `F` may run under autocast.

### Euler (position + velocity)

Carry a speed grid `v` that starts at 0:

```
FORWARD                         BACKWARD
v_new = v_old + h * F(x_old)    x_old = x_new - h * v_new
x_new = x_old + h * v_new       v_old = v_new - h * F(x_old)
```

Default `h = 1/sqrt(L)` (`≈ 0.289` at 12 layers). Same width as the baseline.

### Midpoint / leapfrog

$$p[l+1] = p[l-1] + 2 h\, f_{\theta_l}(p[l])$$

Exact inverse: $p[l-1] = p[l+1] - 2 h\, f_{\theta_l}(p[l])$.

Initialization (lecture notebook): $p[0] = p[1] = \mathrm{Embed}(\text{tokens})$. Default $h=0.5$ so $2h=1$.

Dropout and weight decay are 0 on every run so the comparison is fair.

**T4 AMP trap:** do not use `torch.cuda.is_bf16_supported()`. A T4 reports True but has no bf16 tensor cores. Use compute capability `>= 8` for bf16, else fp16.


## 3. Environment, GPU, and dependency checks


In [ ]:
import os
import sys
from pathlib import Path

# Colab bootstrap: locate helpers/ after a git sparse checkout or ZIP upload.
def find_project_root() -> Path | None:
    def _is_root(path: Path) -> bool:
        return (path / "helpers" / "__init__.py").is_file() and (path / "tests").is_dir()

    seen = set()
    for start in (Path.cwd(), Path("/content"), Path("/content/ERA/Assignment13")):
        try:
            cur = start.resolve()
        except OSError:
            continue
        for candidate in [cur, *cur.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            if _is_root(candidate):
                return candidate
    for root in (Path.cwd(), Path("/content")):
        if not root.exists():
            continue
        for match in root.glob("**/helpers/__init__.py"):
            candidate = match.parent.parent
            if _is_root(candidate):
                return candidate
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    print("helpers/ not found. In Colab, either upload this folder or run:")
    print("  !git clone --depth 1 https://github.com/nitinranjansharma/ERA.git /content/ERA")
    print("  %cd /content/ERA/Assignment13")
else:
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print("Project root:", PROJECT_ROOT)


In [ ]:
# Optional: sparse-checkout only Assignment13 into Colab (uncomment to run).
# %%bash
# mkdir -p /content/ERA && cd /content/ERA
# git init
# git remote add origin https://github.com/nitinranjansharma/ERA.git
# git config core.sparseCheckout true
# echo "Assignment13/*" >> .git/info/sparse-checkout
# git fetch --depth 1 origin main
# git checkout FETCH_HEAD
# ls Assignment13
print("Checkout cell is documentation-only so Run-all does not clone twice.")


In [ ]:
import subprocess
from pathlib import Path

req = Path("requirements.txt")
if req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
# Optional Indic path (proven S13). Harmless if already present.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets>=2.18", "tokenizers>=0.15"])
print("Python", sys.version.split()[0])


In [ ]:
import torch
from helpers.config import (
    get_runtime_info, resolve_device, recommend_precision, TARGET_PARAM_COUNT,
    amp_dtype_for_device,
)

device = resolve_device("auto")
info = get_runtime_info(device)
amp_dtype = amp_dtype_for_device(device)
print(info)
print("Selected device:", device)
print("AMP dtype for F (hidden states stay fp32):", amp_dtype)
print("Target parameter count:", TARGET_PARAM_COUNT)
if device.type == "cuda":
    cap = torch.cuda.get_device_capability()
    print("compute capability:", cap, "(bf16 tensor cores only if major >= 8)")
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    print("No CUDA GPU. Smoke tests are allowed. Full 50M-token A/B/C runs need a Colab GPU.")


## 4. Configuration and random seeds

Set `RUN_MODE` to `"smoke"` or `"full"`. Full mode is refused on CPU unless you override. The three experiment flags default to **off** so opening the notebook does not start a 50M-token job.


In [ ]:
from helpers.config import (
    ExperimentConfig,
    assignment_config,
    set_seed,
    smoke_config,
    tiny_config,
)

RUN_MODE = "smoke"          # "smoke" | "full"
RUN_EXPERIMENT_A = False
RUN_EXPERIMENT_B = False
RUN_EXPERIMENT_C = False
RUN_PILOT = True
RUN_TESTS = True
SEED = 1337                 # proven S13 seed
OUTPUT_DIR = "outputs"
# "sangraha" = Indic 8192 BPE (proven notebook, ~10 min download).
# "wikitext2" = word tokenizer, no extra packages.
# Smoke mode always uses synthetic.
DATASET = "synthetic" if RUN_MODE == "smoke" else "sangraha"

set_seed(SEED)
print("RUN_MODE", RUN_MODE)
print("device", device)
print("seed", SEED)
print("dataset", DATASET)

if RUN_MODE == "full" and device.type != "cuda":
    raise RuntimeError(
        "Full 50M-token runs require CUDA. Set RUN_MODE='smoke' or use a GPU runtime."
    )


## 5. Dataset download or preparation and tokenizer

Full mode can use the proven S13 corpus: AI4Bharat Sangraha (Hindi, Telugu, Tamil, Marathi, Bengali, English) plus an 8192-piece byte-level BPE. Token ids are stored as uint16 `.bin` files and loaded with `np.fromfile` (RAM, not memmap).

Set `DATASET = "wikitext2"` if you cannot stream HuggingFace datasets.


In [ ]:
from helpers.data import prepare_data

download = RUN_MODE == "full"
data_cfg = smoke_config("A") if RUN_MODE == "smoke" else assignment_config("A", dataset=DATASET)
data_cfg = ExperimentConfig.from_dict({**data_cfg.to_dict(), "output_dir": OUTPUT_DIR, "seed": SEED, "dataset": DATASET if RUN_MODE == "full" else "synthetic"})
prepared = prepare_data(data_cfg, download=download)
print("source:", prepared.source)
print("tokenizer vocab:", prepared.tokenizer.vocab_size)
print("train target tokens / epoch:", prepared.train_target_tokens_per_epoch)
print("tokenizer:", prepared.tokenizer_info())

# Diagnostic: data load vs a tiny step (the proven notebook's np.memmap red herring).
import time
batch = next(iter(prepared.train_loader))
t0 = time.time()
for _ in range(5):
    _ = next(iter(prepared.train_loader))
print(f"data only: {(time.time()-t0)/5*1000:.1f} ms/batch")
print("GPU / AMP reminder:", device, amp_dtype)


## 6. Model definition and actual parameter-count verification


In [ ]:
from helpers.config import PARAM_COUNT_TOLERANCE, expected_param_count
from helpers.model import build_model, count_trainable_parameters

full_cfg = assignment_config("A", seed=SEED)
model_a = build_model(full_cfg)
live = count_trainable_parameters(model_a)
formula = expected_param_count(full_cfg)
print("Device:", device)
print(model_a.param_report())
print("live", live, "formula", formula, "tied", full_cfg.tie_embeddings)
assert live == formula
assert abs(live - TARGET_PARAM_COUNT) <= PARAM_COUNT_TOLERANCE * TARGET_PARAM_COUNT
print("Within 15% of 20M:", True)
del model_a


## 7. Unit tests and reversible numerical reconstruction checks


In [ ]:
import pytest
from helpers.model import build_model
from helpers.reversible import verify_midpoint_reconstruction

if RUN_TESTS:
    rc = pytest.main(["-q", "tests"])
    print("pytest exit code", rc)
else:
    print("RUN_TESTS is False; skipped.")

tiny = tiny_config(experiment="B", variant="midpoint")
rev = build_model(tiny).eval()
import torch
x = torch.randn(2, tiny.block_size, tiny.n_embd)
report = verify_midpoint_reconstruction(x, rev.blocks, two_h=tiny.two_h, atol=5e-4)
print(report.to_dict())


## 7b. Three proofs (lecture notebook)

These are the checks the proven S13 write-up treats as the point of the assignment:

1. **Invert** — float64 round-trip of both state grids (Euler velocity and midpoint pair).
2. **Gradients** — memory-free backward vs storing-all-activations, tensor by tensor.
3. **Memory vs depth** — peak extra CUDA memory at 4/8/16/32 layers. Skipped on CPU.

Also: AMP dtype is chosen from **compute capability** (`>= 8` → bf16, else fp16). `is_bf16_supported()` is True on a T4 and is a ~5× slowdown trap.


In [ ]:
from helpers.config import amp_dtype_for_device, get_runtime_info
from helpers.proofs import proof_gradients_match, proof_invert_stack, proof_memory_vs_depth

print("AMP dtype for F (hidden states stay fp32):", amp_dtype_for_device(device))
print("runtime", {k: get_runtime_info(device)[k] for k in ("gpu_name", "gpu_capability", "recommended_precision")})

p1 = proof_invert_stack()
print("PROOF 1 invert")
for row in p1:
    print(row)
assert all(r["ok"] for r in p1)

p2 = proof_gradients_match()
print("PROOF 2 grads")
for row in p2:
    print(row)
assert all(r["ok"] for r in p2)

p3 = proof_memory_vs_depth()
print("PROOF 3 memory vs depth")
print(p3)



## 8. Baseline pilot and fixed-batch selection


In [ ]:
from helpers.benchmark import measure_training_step, make_step_probe, search_max_micro_batch
from helpers.model import build_model

FIXED_MICRO_BATCH = 4 if RUN_MODE == "smoke" else 32
print("Pilot micro-batch (runs 1-3):", FIXED_MICRO_BATCH)
print("Run 4 will try 128, then 64, then 32")

if RUN_PILOT:
    cfg = smoke_config("A") if RUN_MODE == "smoke" else assignment_config("A", micro_batch_size=FIXED_MICRO_BATCH, dataset=DATASET)
    cfg = ExperimentConfig.from_dict({**cfg.to_dict(), "micro_batch_size": FIXED_MICRO_BATCH, "seed": SEED})
    model = build_model(cfg).to(device)
    probe = make_step_probe(model, cfg, device)

    def timed():
        probe(FIXED_MICRO_BATCH)
        return FIXED_MICRO_BATCH * cfg.block_size

    timing = measure_training_step(
        timed,
        warmup_steps=1,
        measured_steps=2,
        device=device,
        micro_batch_size=FIXED_MICRO_BATCH,
        sequence_length=cfg.block_size,
    )
    print(timing.to_dict())
    del model
else:
    print("RUN_PILOT is False; using FIXED_MICRO_BATCH as configured.")


## 9. Experiment A: standard training

Ordinary residual Transformer, fixed micro-batch, 50M target tokens in full mode. Smoke mode uses the tiny/smoke config and a few steps.


In [ ]:
from helpers.train import train
from helpers.data import prepare_data

reports = {}
histories = {}

def run_experiment(name, cfg):
    payload = {**cfg.to_dict(), "output_dir": OUTPUT_DIR, "seed": SEED, "device": str(device)}
    if RUN_MODE == "full":
        payload["dataset"] = DATASET
    cfg = ExperimentConfig.from_dict(payload)
    data = prepare_data(cfg, download=(RUN_MODE == "full" and cfg.dataset in ("wikitext2", "sangraha")))
    report = train(cfg, data=data)
    reports[name] = report
    histories[name] = report.get("extra", {}).get("history") or []
    keys = ("experiment", "variant", "status", "trainable_parameters", "micro_batch_size",
            "training_target_tokens", "final_train_loss", "final_val_loss")
    print({k: report.get(k) for k in keys})
    extra = report.get("extra") or {}
    print("perplexity", extra.get("perplexity"), "round-trip", extra.get("round_trip_error"))
    return report

if RUN_EXPERIMENT_A:
    cfg = smoke_config("A") if RUN_MODE == "smoke" else assignment_config("A", micro_batch_size=FIXED_MICRO_BATCH, dataset=DATASET)
    run_experiment("A", cfg)
else:
    print("Experiment A skipped. Set RUN_EXPERIMENT_A = True to run.")


## 10. Experiment B: reversible fixed-batch training (Euler and midpoint)

The lecture asks you to test **both** numerical variants at the same micro-batch as A.
Euler uses position+velocity with `h = 1/sqrt(L)`. Midpoint uses `h = 0.5`.
Set `RUN_EXPERIMENT_B = True` (and `RUN_MODE = "full"` for 50M tokens).


In [ ]:
if RUN_EXPERIMENT_B:
    factory = smoke_config if RUN_MODE == "smoke" else assignment_config
    for variant in ("euler", "midpoint"):
        cfg = factory("B", variant=variant, micro_batch_size=FIXED_MICRO_BATCH)
        run_experiment(f"B_{variant}", cfg)
    if "B_euler" in reports and "B_midpoint" in reports:
        winner = min(("euler", "midpoint"), key=lambda v: reports[f"B_{v}"].get("final_val_loss") or reports[f"B_{v}"].get("final_train_loss") or 1e9)
        print("fixed-batch winner:", winner)
else:
    print("Experiment B skipped. Set RUN_EXPERIMENT_B = True to run both Euler and midpoint.")


## 11. Experiment C: push the winner's batch (128, then 64, then 32)

Same reversible implementation as the better of Euler vs midpoint at the fixed batch.
Try micro-batch 128, fall back on OOM, optionally scale LR by `sqrt(new/old)`.
The 50M-token budget is unchanged, so fewer optimizer steps are expected and loss may worsen.


In [ ]:
import math
import torch

if RUN_EXPERIMENT_C:
    winner = "midpoint"
    if "B_euler" in reports and "B_midpoint" in reports:
        winner = min(
            ("euler", "midpoint"),
            key=lambda v: reports[f"B_{v}"].get("final_val_loss")
            or reports[f"B_{v}"].get("final_train_loss")
            or 1e9,
        )
    print("C uses winner:", winner)
    factory = smoke_config if RUN_MODE == "smoke" else assignment_config
    batches = (8, 4) if RUN_MODE == "smoke" else (128, 64, 32)
    for batch in batches:
        try:
            lr = 3e-4 * math.sqrt(batch / max(FIXED_MICRO_BATCH, 1))
            cfg = factory("C", variant=winner, micro_batch_size=batch, learning_rate=lr, dataset=DATASET if RUN_MODE == "full" else "synthetic")
            run_experiment(f"C_{winner}_{batch}", cfg)
            break
        except torch.cuda.OutOfMemoryError:
            print(f"batch {batch} OOM, trying smaller")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
else:
    print("Experiment C skipped. Set RUN_EXPERIMENT_C = True to push the winner's batch.")


## 12. Loss, throughput, and memory plots


In [ ]:
from helpers.reporting import (
    export_run_artifacts, markdown_table, pending_results_template,
    plot_four_run_panel, plot_euler_vs_midpoint,
)

completed = list(reports.values())
if completed:
    artifacts = export_run_artifacts(OUTPUT_DIR, completed, histories)
    print(artifacts)
    print(markdown_table(completed))
    named = reports
    plot_four_run_panel(named, f"{OUTPUT_DIR}/plots/results.png")
    if "B_euler" in named and "B_midpoint" in named:
        plot_euler_vs_midpoint(named["B_euler"], named["B_midpoint"], f"{OUTPUT_DIR}/plots/euler_vs_midpoint.png")
else:
    print("No completed runs in this session. Pending template:")
    print(markdown_table(pending_results_template()))


## 13. Comparison tables and experiment validation


In [ ]:
from helpers.metrics import validate_report
from helpers.reporting import markdown_table

if completed:
    for report in completed:
        validate_report(report)
        print("validated", report["experiment"], report["status"])
    print(markdown_table(completed))
else:
    print("Nothing to validate. Full A/B/C metrics appear here only after those cells run.")


## 14. Interpretation, limitations, and conclusion

Fill this section from **actual** `outputs/results.json` after the four runs complete. Until then:

- Reversibility trades activation memory for extra compute (one recomputation of $F_l$ per layer in backward).
- Run 4's larger batch buys fewer optimizer steps on a fixed 50M-token budget. Loss may get worse; that is the finding, not a bug.
- Euler and midpoint use different default $h$. A gap between them is not a pure integrator bake-off.
- Under 16-bit AMP, round-trip error is ~1e-2, not the Proof 1 figure of ~1e-16. Report both.
- Do not use `torch.cuda.is_bf16_supported()` on a T4.
- This authoring environment is CPU-only: peak GPU memory is `None`, and 50M-token numbers are pending. Do not paste another notebook's T4 table here.
- Dropout and weight decay are 0 on every run so the comparison is fair.


## 15. Export of CSV, JSON, configuration, and plots


In [ ]:
from pathlib import Path
import json

out = Path(OUTPUT_DIR)
print("output dir", out.resolve())
for path in sorted(out.rglob("*")):
    if path.is_file() and path.name != ".gitkeep":
        print(path.relative_to(out), path.stat().st_size, "bytes")

# Never invent 50M-token numbers. If results.json is missing, write a pending template only.
results_path = out / "results.json"
if not results_path.exists():
    from helpers.reporting import write_json, write_comparison_table, pending_results_template
    pending = pending_results_template()
    write_json(results_path, pending)
    write_comparison_table(out / "results.csv", pending)
    print("Wrote pending results template (not experimental measurements).")
else:
    print("Existing results.json:")
    print(json.dumps(json.loads(results_path.read_text())[:1], indent=2)[:800])
